In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import os
from viz_config import VizConfig

# ==========================================
# 0. Global configuration and style setup
# ==========================================
# Load the paper-style configuration from VizConfig
VizConfig.set_style()

# Pull constants from VizConfig for local reuse
TITLE_SIZE = VizConfig.TITLE_SIZE
LABEL_SIZE = VizConfig.LABEL_SIZE
TICK_SIZE = VizConfig.TICK_SIZE
LEGEND_SIZE = VizConfig.LEGEND_SIZE

OUTPUT_DIR = "Refined_Results_v4"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 1. Data loading
# ==========================================
csv_path = os.path.join(OUTPUT_DIR, "final_summary_r2.csv")

# Check whether the real data file exists
if not os.path.exists(csv_path):
    print(f"Note: {csv_path} missing - using synthetic data for the demo.")
    # Build synthetic data to show the figure appearance
    # Synthetic noise levels from 0% to 200%
    x_raw = np.linspace(0, 2.0, 21) 
    
    # Synthetic baseline (Direct SR): sigmoid drop around 50% noise
    y_direct = 0.99 - (0.99 / (1 + np.exp(-10 * (x_raw - 0.5)))) - 0.1 * x_raw
    y_direct[y_direct < -0.1] = -0.1 # Floor value to keep the curve reasonable
    
    # Synthetic Ours (hybrid framework): slow linear decline - robust
    y_hybrid = 0.99 - 0.15 * x_raw 
else:
    # Load real experimental data
    df = pd.read_csv(csv_path)
    x_raw = df['Noise_Ratio'].values
    y_direct = df['R2_Direct_PySR'].values
    y_hybrid = df['R2_Hybrid_PySR'].values

# ==========================================
# 2. Plot initialisation
# ==========================================
fig, ax = plt.subplots(figsize=(10, 7))

# Pin the critical breakdown point used to split the zones
cliff_x = 0.475 
max_x = 2.0     # XMax visible axis range

# ==========================================
# 3. Plot background zones and threshold
# ==========================================
# A. Coloured background blocks to separate"robust zone" and "failure zone"
# Left: robust zone (light green)
ax.axvspan(xmin=-0.05, xmax=cliff_x, color=VizConfig.COLOR_SUCCESS, alpha=0.08, zorder=0) 
# Right: failure zone (light red)
ax.axvspan(xmin=cliff_x, xmax=max_x+0.1, color=VizConfig.COLOR_HIGHLIGHT, alpha=0.08, zorder=0)

# B. Draw the vertical divider
ax.axvline(x=cliff_x, color=VizConfig.COLOR_SECONDARY, linestyle='--', linewidth=1.5, zorder=1)
# Annotate the divider with text
ax.text(x=cliff_x, y=0.5, s="Critical Breakdown Point", 
        color=VizConfig.COLOR_AXIS, fontsize=12, ha='center', va='bottom', fontweight='bold',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7)) # White bbox so the text does not clash with the line

# ==========================================
# 4. Plot the data curves
# ==========================================
# A. Baseline: Direct SR
# Secondary colour (grey) + dashed - signals the control group
# Edge-only markers with white centres for depth
line_base, = ax.plot(x_raw, y_direct, marker='o', markersize=8, linewidth=2.5, 
                     color=VizConfig.COLOR_SECONDARY, linestyle='--', label='Baseline (Direct SR)', zorder=2,
                     markeredgecolor='white', markeredgewidth=1)

# B. Ours: Hybrid Framework
# COLOR_MAIN (deep blue) + solid - emphasises our proposed method
# Star markers to stand out
line_ours, = ax.plot(x_raw, y_hybrid, marker='*', markersize=12, linewidth=2.5, 
                     color=VizConfig.COLOR_MAIN, linestyle='-', label='Ours (Hybrid Framework)', zorder=3,
                     markeredgecolor='white', markeredgewidth=1)

# ==========================================
# 6. Add helper annotations
# ==========================================
# A. Zone-name labels (bottom)
text_y_pos = 0.05 
# "Robust Zone" (green)
ax.text(x=cliff_x/2, y=text_y_pos, s="Robust Zone", 
        color=VizConfig.COLOR_SUCCESS, fontsize=VizConfig.LABEL_SIZE, fontweight='bold', ha='center', va='bottom')
# "Failure Zone" (red)
ax.text(x=cliff_x + (max_x - cliff_x)/2, y=text_y_pos, s="Failure Zone", 
        color=VizConfig.COLOR_HIGHLIGHT, fontsize=VizConfig.LABEL_SIZE, fontweight='bold', ha='center', va='bottom')

# B. Engineering-requirement line
# Horizontal dashed line at R^2 = 0.9 - engineering-acceptable threshold
ax.axhline(y=0.9, color=VizConfig.COLOR_AXIS, linestyle=':', linewidth=2, zorder=3)
ax.text(x=0.03, y=0.85, s="Engineering Requirement", fontsize=12, fontweight='bold', color=VizConfig.COLOR_AXIS)

# ==========================================
# 7. Axes and legend
# ==========================================
# A. XTick configuration
# One tick every 20%
x_ticks = np.arange(0, 2.1, 0.2)
# Format values as a percent string (e.g. "20%")
x_labels = [f"{int(val*100)}%" for val in x_ticks]
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, rotation=0, fontsize=TICK_SIZE)
ax.tick_params(axis='y', labelsize=TICK_SIZE)

# B. Title and labels
ax.set_xlabel("Noise Level (%)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel(r"Coefficient of Determination ($R^2$)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_title("Robustness Comparison: Hybrid vs. Direct", fontsize=TITLE_SIZE, pad=35, fontweight='bold')

# C. Legend
# Top-right, with frame
ax.legend(fontsize=LEGEND_SIZE, loc='upper right', frameon=True, facecolor='white', framealpha=1) 

# D. Axis limits and grid
ax.set_xlim(0, 2.05)
ax.set_ylim(-0.05, 1.1) 
ax.grid(True, linestyle='--', alpha=0.4, color=VizConfig.COLOR_GRID, zorder=1)

# ==========================================
# 8. Save and export
# ==========================================
# tight_layout with pad_inches so margins are preserved
plt.tight_layout()
output_path = os.path.join(OUTPUT_DIR, "6.pdf")
# Save as a high-resolution PDF
plt.savefig(output_path, dpi=VizConfig.DPI, bbox_inches='tight', pad_inches=0.1) 

print(f"Comparison figure saved to: {output_path}")
plt.show()